# DeepSeek API - Примеры использования

DeepSeek - дешёвая альтернатива OpenAI/Claude с совместимым API.

**Документация:** https://platform.deepseek.com/api-docs

In [1]:
# Установка OpenAI SDK (DeepSeek совместим с ним)
# !pip install openai

## 1. Настройка клиента

In [2]:
from dotenv import load_dotenv
import os
from openai import OpenAI

# Загружаем ключи из .env файла (в корне проекта)
load_dotenv("../../.env")

# DeepSeek использует OpenAI-совместимый API
client = OpenAI(
    api_key=os.getenv('DEEPSEEK_API_KEY'),
    base_url="https://api.deepseek.com"
)

print("Клиент настроен")

Клиент настроен


## 2. Простой запрос

In [3]:
response = client.chat.completions.create(
    model="deepseek-chat",  # или deepseek-coder для кода
    messages=[
        {"role": "user", "content": "Привет! Объясни в 2 предложениях, что такое машинное обучение."}
    ],
    max_tokens=500
)

print(response.choices[0].message.content)

Машинное обучение — это раздел искусственного интеллекта, который позволяет компьютерам обучаться выполнять задачи на основе данных, а не явных инструкций. Вместо программирования каждого шага алгоритмы находят в информации закономерности и самостоятельно строят модели для прогнозирования или принятия решений.


## 3. Системный промпт

In [ ]:
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "Ты - опытный Python разработчик. Отвечай кратко с примерами кода."},
        {"role": "user", "content": "Как прочитать JSON файл?"}
    ],
    max_tokens=500
)

print(response.choices[0].message.content)

## 4. Диалог (Multi-turn)

In [ ]:
conversation = [
    {"role": "user", "content": "Меня зовут Руслан, я изучаю AI."},
    {"role": "assistant", "content": "Приятно познакомиться, Руслан! AI - увлекательная область."},
    {"role": "user", "content": "Как меня зовут?"}
]

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=conversation,
    max_tokens=100
)

print(response.choices[0].message.content)

## 5. Streaming (потоковый вывод)

In [ ]:
print("Потоковый вывод:")
print("-" * 40)

stream = client.chat.completions.create(
    model="deepseek-chat",
    messages=[{"role": "user", "content": "Напиши короткое стихотворение про Python."}],
    max_tokens=200,
    stream=True
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

print("\n" + "-" * 40)

## 6. JSON вывод

In [ ]:
import json

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "Отвечай только валидным JSON."},
        {"role": "user", "content": """Проанализируй текст и верни JSON:
        {"sentiment": "positive/negative/neutral", "topics": ["тема1"], "summary": "краткое содержание"}
        
        Текст: Сегодня я научился работать с API, это было интересно!"""}
    ],
    max_tokens=300
)

result = json.loads(response.choices[0].message.content)
print(json.dumps(result, ensure_ascii=False, indent=2))

## 7. DeepSeek Coder (специализация на коде)

In [ ]:
response = client.chat.completions.create(
    model="deepseek-coder",  # Модель для кода
    messages=[
        {"role": "system", "content": "Ты - эксперт по Python. Пиши чистый, документированный код."},
        {"role": "user", "content": "Напиши функцию для проверки, является ли строка палиндромом."}
    ],
    max_tokens=500
)

print(response.choices[0].message.content)

## 8. Подсчёт токенов и стоимости

In [ ]:
# Информация о последнем запросе
print(f"Входные токены: {response.usage.prompt_tokens}")
print(f"Выходные токены: {response.usage.completion_tokens}")
print(f"Всего токенов: {response.usage.total_tokens}")

# Цены DeepSeek (очень дёшево!)
# deepseek-chat: $0.14 / 1M input, $0.28 / 1M output
# deepseek-coder: $0.14 / 1M input, $0.28 / 1M output

input_cost = response.usage.prompt_tokens * 0.14 / 1_000_000
output_cost = response.usage.completion_tokens * 0.28 / 1_000_000
total = input_cost + output_cost

print(f"\nСтоимость запроса: ${total:.8f}")
print(f"Это примерно {total * 100:.6f} рублей")

## 9. Сравнение цен

| Модель | Input (1M tokens) | Output (1M tokens) |
|--------|-------------------|--------------------|
| DeepSeek Chat | $0.14 | $0.28 |
| GPT-4 Turbo | $10 | $30 |
| Claude 3.5 Sonnet | $3 | $15 |
| GPT-3.5 Turbo | $0.50 | $1.50 |

**DeepSeek в ~20-100 раз дешевле конкурентов!**

## 10. Практический пример: Переводчик

In [ ]:
def translate(text: str, target_lang: str = "русский") -> str:
    """Переводит текст на указанный язык."""
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": f"Ты - переводчик. Переводи текст на {target_lang}. Выводи только перевод."},
            {"role": "user", "content": text}
        ],
        max_tokens=1000
    )
    return response.choices[0].message.content

# Тест
english_text = "Machine learning is a subset of artificial intelligence that enables systems to learn from data."
russian = translate(english_text, "русский")
print(f"Оригинал: {english_text}")
print(f"Перевод: {russian}")

## 11. Интеграция с Telegram (пример)

In [ ]:
import requests

# Токены уже загружены из .env
TELEGRAM_TOKEN = os.getenv('TELEGRAM_TOKEN')
CHAT_ID = os.getenv('TELEGRAM_CHAT_ID')

def send_telegram(text: str):
    """Отправляет сообщение в Telegram."""
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
    response = requests.post(url, json={
        "chat_id": CHAT_ID,
        "text": text
    })
    return response.json()

def ask_ai_and_send(question: str):
    """Спрашивает AI и отправляет ответ в Telegram."""
    # Получаем ответ от DeepSeek
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": question}],
        max_tokens=500
    )
    answer = response.choices[0].message.content
    
    # Отправляем в Telegram
    send_telegram(f"Вопрос: {question}\n\nОтвет: {answer}")
    return answer

# Тест - отправит сообщение в твой Telegram!
# ask_ai_and_send("Что такое Python в одном предложении?")

## Доступные модели DeepSeek

| Модель | Описание |
|--------|----------|
| `deepseek-chat` | Универсальная модель для диалогов |
| `deepseek-coder` | Специализация на коде |
| `deepseek-reasoner` | Для сложных рассуждений |